
# SU(2) plaquette evolution — promise-aware / don't-care low-\(T\) circuit

This notebook implements the next optimization of the \(j_{\max}=\tfrac12\)
SU(2) plaquette evolution.

The 12 data qubits span \(2^{12}=4096\) computational states, but only
\(5^4=625\) encode valid four-link states.  Therefore the circuit is required
to reproduce the plaquette evolution only on the 625-dimensional encoded
subspace.  Its action on the remaining 3471 dimensions is a **don't-care**.

The main promise-aware change is:

- in the \(D=4\) domain-wall sector, **every encoded state is bright**;
  there are no encoded dark states there;
- therefore the \(g=\tfrac14\) dynamical rotation no longer needs the
  global `active` control;
- this deliberately allows the circuit to act on some invalid codewords.

Two additional \(T\)-oriented structural improvements are included:

1. after the complementary-\(j\) pair transform, the three coupling classes
   \(g=1,\frac12,\frac14\) are computed directly from only three relative
   \(j\)-bits;
2. the four domain-color invariance Toffolis are replaced by
   relative-phase Toffolis (`RCCX`) with a swapped control ordering.
   On the physical bright subspace this changes only the sign of the
   \(D=2\) blocks, which is corrected by one Clifford `CZ`.

The exact discrete-logic \(T\)-count before synthesizing the arbitrary
rotations drops from 328 in the previous notebook to

\[
\boxed{296}
\]

and the dynamical rotations change from

\[
3\times C^2R_x
\]

to

\[
2\times C^2R_x + 1\times CR_x.
\]

The notebook includes exhaustive classical checks of the 625-state promise.


In [5]:

import numpy as np

from qiskit import QuantumCircuit, QuantumRegister

try:
    import qiskit
    print("Qiskit version:", qiskit.__version__)
except Exception:
    pass

TOL = 1.0e-12

import sys
!{sys.executable} -m pip install pylatexenc


Qiskit version: 2.5.2

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip



## 1. Three-qubit link encoding

For every link,

\[
|000\rangle = |j=0\rangle,
\]

and

\[
|100\rangle,|101\rangle,|110\rangle,|111\rangle
\]

encode the four \(j=\tfrac12\) states.

The unused one-link codewords are

\[
001,\;010,\;011.
\]

Within a link the Qiskit qubit order is

\[
(q_{m'},q_m,q_j).
\]


In [6]:

def qmp(link):
    return 3 * link

def qm(link):
    return 3 * link + 1

def qj(link):
    return 3 * link + 2


JIDX = [qj(l) for l in range(4)]

# Vertices a,b,c,d.
# Each tuple is (x endpoint bit, y endpoint bit, incident jx, incident jy).
VERTEX_SPEC = [
    (qm(3),  qm(0),  qj(3), qj(0)),   # a
    (qmp(0), qm(1),  qj(0), qj(1)),   # b
    (qmp(1), qmp(2), qj(1), qj(2)),   # c
    (qm(2),  qmp(3), qj(2), qj(3)),   # d
]



## 2. Exact controlled-H used in the bright-state compression

The controlled Hadamard itself is implemented exactly with two \(T\)-type
gates.  The condition \(j_xj_y\) is computed temporarily with an `RCCX`,
used only as a control, and then uncomputed.


In [7]:

def controlled_h_exact(qc, control, target):
    # Exact controlled-H:
    # Sdg H Tdg --CX-- T H S  on the target.
    qc.sdg(target)
    qc.h(target)
    qc.tdg(target)
    qc.cx(control, target)
    qc.t(target)
    qc.h(target)
    qc.s(target)


def conditional_h_on_11(qc, jx, jy, target, scratch):
    qc.rccx(jx, jy, scratch)
    controlled_h_exact(qc, scratch, target)
    qc.rccx(jx, jy, scratch)



## 3. Registers

The new circuit uses 34 qubits:

- 12 data qubits;
- 4 domain-wall bits `p`;
- 4 error bits `e`;
- 7 work bits for the `active` AND chain;
- 3 coupling-class flags `g`;
- 3 temporary bits;
- 1 scratch bit for the controlled-H construction.

The workspace is clean at the beginning and is returned to zero at the end.


In [8]:

def make_registers():
    d = QuantumRegister(12, "d")
    p = QuantumRegister(4,  "p")
    e = QuantumRegister(4,  "e")
    w = QuantumRegister(7,  "w")
    g = QuantumRegister(3,  "g")
    tmp = QuantumRegister(3, "tmp")
    h = QuantumRegister(1,  "h")

    qc = QuantumCircuit(d, p, e, w, g, tmp, h, name="UB_dontcare_lowT")

    return qc, {
        "d": d,
        "p": p,
        "e": e,
        "w": w,
        "g": g,
        "tmp": tmp,
        "h": h,
    }


def _vertex_qubits(regs):
    d = regs["d"]
    return [
        (d[x], d[y], d[jx], d[jy])
        for (x, y, jx, jy) in VERTEX_SPEC
    ]



## 4. Bright/color basis transformation

This is the same exact physical bright-state transform used in the previous
optimization level.  We deliberately keep the exact `CCX` here: replacing
these particular gates by relative-phase Toffolis introduces
color-dependent \(\pm i\) phases that do not reduce the total \(T\)-count
without further non-Clifford phase corrections.


In [9]:

def forward_color_transform(qc, regs):
    d = regs["d"]
    h = regs["h"][0]
    verts = _vertex_qubits(regs)

    # U^\dagger links 2 and 3: normalize their occupied endpoint colors.
    for link in (2, 3):
        qc.cx(d[qj(link)], d[qm(link)])
        qc.cx(d[qj(link)], d[qmp(link)])

    # Canonical vertex variables.
    for x, y, jx, jy in verts:
        qc.cx(x, y)
        qc.ccx(jx, y, x)

    # Phase convention for the plaquette tensor.
    qc.z(verts[0][0])
    qc.z(verts[2][0])

    # Compress the equal-weight bright combinations.
    for x, y, jx, jy in verts:
        conditional_h_on_11(qc, jx, jy, x, h)


def inverse_color_transform(qc, regs):
    d = regs["d"]
    h = regs["h"][0]
    verts = _vertex_qubits(regs)

    for x, y, jx, jy in reversed(verts):
        conditional_h_on_11(qc, jx, jy, x, h)

    qc.z(verts[2][0])
    qc.z(verts[0][0])

    for x, y, jx, jy in reversed(verts):
        qc.ccx(jx, y, x)
        qc.cx(x, y)

    for link in (3, 2):
        qc.cx(d[qj(link)], d[qmp(link)])
        qc.cx(d[qj(link)], d[qm(link)])



## 5. Domain walls and physical bright selector

Before the complementary-\(j\) pair transform define

\[
\begin{aligned}
p_0 &= j_3\oplus j_0,\\
p_1 &= j_0\oplus j_1,\\
p_2 &= j_1\oplus j_2,\\
p_3 &= j_2\oplus j_3.
\end{aligned}
\]

A physical transformed root is bright precisely when

- all four compressed internal-color bits \(x_i\) vanish;
- whenever \(p_i=0\), the mismatch bit \(y_i\) also vanishes.

The latter condition is stored as

\[
e_i=(1-p_i)y_i.
\]

These checks cannot in general be removed: they distinguish physical dark
states, not merely unused codewords.


In [10]:

def compute_domain_walls(qc, regs):
    d = regs["d"]
    p = regs["p"]

    j = [d[i] for i in JIDX]

    pairs = [
        (j[3], j[0]),
        (j[0], j[1]),
        (j[1], j[2]),
        (j[2], j[3]),
    ]

    for pi, (a, b) in zip(p, pairs):
        qc.cx(a, pi)
        qc.cx(b, pi)


def uncompute_domain_walls(qc, regs):
    d = regs["d"]
    p = regs["p"]

    j = [d[i] for i in JIDX]

    pairs = [
        (j[3], j[0]),
        (j[0], j[1]),
        (j[1], j[2]),
        (j[2], j[3]),
    ]

    for pi, (a, b) in reversed(list(zip(p, pairs))):
        qc.cx(b, pi)
        qc.cx(a, pi)


def compute_errors(qc, regs):
    p = regs["p"]
    e = regs["e"]
    ys = [v[1] for v in _vertex_qubits(regs)]

    for pi, yi, ei in zip(p, ys, e):
        qc.x(pi)
        qc.rccx(pi, yi, ei)
        qc.x(pi)


def uncompute_errors(qc, regs):
    p = regs["p"]
    e = regs["e"]
    ys = [v[1] for v in _vertex_qubits(regs)]

    for pi, yi, ei in reversed(list(zip(p, ys, e))):
        qc.x(pi)
        qc.rccx(pi, yi, ei)
        qc.x(pi)


def compute_active_flag(qc, regs):
    e = regs["e"]
    w = regs["w"]
    xs = [v[0] for v in _vertex_qubits(regs)]

    conditions = xs + list(e)

    # Negative controls: require all eight conditions to be zero.
    for q in conditions:
        qc.x(q)

    qc.rccx(conditions[0], conditions[1], w[0])

    for i in range(2, 8):
        qc.rccx(w[i - 2], conditions[i], w[i - 1])

    # w[6] is the active-bright flag.


def uncompute_active_flag(qc, regs):
    e = regs["e"]
    w = regs["w"]
    xs = [v[0] for v in _vertex_qubits(regs)]

    conditions = xs + list(e)

    for i in reversed(range(2, 8)):
        qc.rccx(w[i - 2], conditions[i], w[i - 1])

    qc.rccx(conditions[0], conditions[1], w[0])

    for q in reversed(conditions):
        qc.x(q)



## 6. Complementary-\(j\) pair basis

Take \(j_3\) as the pair/side qubit.  The three CNOTs

\[
j_k\longrightarrow j_k\oplus j_3,\qquad k=0,1,2
\]

make \(j_0,j_1,j_2\) relative labels, while \(j_3\) is the only bit that
distinguishes the two complementary members of a bright pair.


In [11]:

def pair_basis_transform(qc, regs):
    d = regs["d"]
    side = d[JIDX[3]]

    for k in (0, 1, 2):
        qc.cx(side, d[JIDX[k]])


def inverse_pair_basis_transform(qc, regs):
    d = regs["d"]
    side = d[JIDX[3]]

    for k in (2, 1, 0):
        qc.cx(side, d[JIDX[k]])



## 7. Coupling class from only three relative \(j\)-bits

After the pair transformation the three relative bits are

\[
r_0=j_0\oplus j_3,\qquad
r_1=j_1\oplus j_3,\qquad
r_2=j_2\oplus j_3.
\]

Then

\[
D=0 \iff (r_0r_1r_2)=000,
\]

and

\[
D=4 \iff (r_0r_1r_2)=101.
\]

All other six patterns have \(D=2\).

Thus:

- `g[0]` = \(g=1\);
- `g[1]` = \(g=\tfrac14\);
- `g[2]` = \(g=\tfrac12\).

This costs 8 `RCCX` gates for compute + uncompute, rather than 12 in the
previous implementation.


In [12]:

def compute_g_class_relative(qc, regs):
    d = regs["d"]
    g = regs["g"]
    tmp = regs["tmp"]

    r0 = d[JIDX[0]]
    r1 = d[JIDX[1]]
    r2 = d[JIDX[2]]

    g1, gquarter, ghalf = g
    t0, t1, _ = tmp

    # g = 1: relative pattern 000.
    qc.x(r0)
    qc.x(r1)
    qc.x(r2)

    qc.rccx(r0, r1, t0)
    qc.rccx(t0, r2, g1)

    qc.x(r2)
    qc.x(r1)
    qc.x(r0)

    # g = 1/4: relative pattern 101.
    qc.x(r1)

    qc.rccx(r0, r1, t1)
    qc.rccx(t1, r2, gquarter)

    qc.x(r1)

    # The remaining six relative patterns all have g = 1/2.
    qc.x(ghalf)
    qc.cx(g1, ghalf)
    qc.cx(gquarter, ghalf)


def uncompute_g_class_relative(qc, regs):
    d = regs["d"]
    g = regs["g"]
    tmp = regs["tmp"]

    r0 = d[JIDX[0]]
    r1 = d[JIDX[1]]
    r2 = d[JIDX[2]]

    g1, gquarter, ghalf = g
    t0, t1, _ = tmp

    qc.cx(gquarter, ghalf)
    qc.cx(g1, ghalf)
    qc.x(ghalf)

    # Undo g = 1/4.
    qc.x(r1)

    qc.rccx(t1, r2, gquarter)
    qc.rccx(r0, r1, t1)

    qc.x(r1)

    # Undo g = 1.
    qc.x(r0)
    qc.x(r1)
    qc.x(r2)

    qc.rccx(t0, r2, g1)
    qc.rccx(r0, r1, t0)

    qc.x(r2)
    qc.x(r1)
    qc.x(r0)



## 8. Promise-aware domain-color invariance

The two members of a bright pair differ in the domain-wall color \(y_i\)
whenever \(p_i=1\).  The previous circuit used exact Toffolis

\[
y_i\leftarrow y_i\oplus(j_3p_i).
\]

Here they are replaced by `RCCX(p_i, j_3, y_i)`, **with the domain-wall bit
as the first control and the side bit as the second control**.

This ordering matters.  On the 41 physical bright pairs, the relative phase
introduced by these four RCCX gates is

\[
(-1)^{D/2}.
\]

Therefore it is:

- \(+1\) for \(D=0\);
- \(-1\) for \(D=2\);
- \(+1\) for \(D=4\).

One Clifford `CZ(ghalf, side)` corrects it exactly.

The permutation action is the same as the four exact Toffolis, but the
fixed cost is \(8\times4T=32T\) rather than \(8\times7T=56T\).


In [13]:

def make_domain_colors_invariant_rccx(qc, regs):
    d = regs["d"]
    p = regs["p"]
    side = d[JIDX[3]]

    ys = [v[1] for v in _vertex_qubits(regs)]

    # IMPORTANT: control ordering is (p_i, side), not (side, p_i).
    for pi, yi in zip(p, ys):
        qc.rccx(pi, side, yi)


def undo_domain_colors_invariant_rccx(qc, regs):
    d = regs["d"]
    p = regs["p"]
    side = d[JIDX[3]]

    ys = [v[1] for v in _vertex_qubits(regs)]

    for pi, yi in reversed(list(zip(p, ys))):
        qc.rccx(pi, side, yi)



## 9. Bright-pair sign

For the exact basis convention the residual sign was

\[
(-1)^{y_1+y_3+p_1+p_2p_3}.
\]

The swapped-control RCCX domain transformation adds one further minus sign
for the \(D=2\) class.

We therefore use

\[
(-1)^{
y_1+y_3+p_1+p_2p_3+I_{D=2}
}.
\]

The \(p_2p_3\) product is kept in `tmp[2]` across the central rotations.


In [14]:

def compute_sign_tmp(qc, regs):
    p = regs["p"]
    tmp = regs["tmp"]

    qc.rccx(p[2], p[3], tmp[2])


def uncompute_sign_tmp(qc, regs):
    p = regs["p"]
    tmp = regs["tmp"]

    qc.rccx(p[2], p[3], tmp[2])


def apply_sign_conjugation(qc, regs):
    d = regs["d"]
    p = regs["p"]
    g = regs["g"]
    tmp = regs["tmp"]

    side = d[JIDX[3]]
    ys = [v[1] for v in _vertex_qubits(regs)]

    _, _, ghalf = g

    qc.cz(ys[1], side)
    qc.cz(ys[3], side)
    qc.cz(p[1], side)
    qc.cz(tmp[2], side)

    # Correction for the swapped-control RCCX domain transform.
    qc.cz(ghalf, side)



## 10. The promise-aware dynamical rotations

Let

\[
\alpha=\frac{2dt}{g_sg_ta}.
\]

The three bright couplings require

\[
\theta_x=-2\alpha g.
\]

For \(D=0\) and \(D=2\), physical dark states exist, so the global `active`
control remains essential.

For \(D=4\), however, **all physical roots are bright**.  Therefore the
\(g=\tfrac14\) rotation is controlled only by the \(D=4\) class flag:

```python
qc.crx(..., gquarter, side)
```

rather than

```python
qc.mcrx(..., [active, gquarter], side)
```

This is the explicit use of the 3471-dimensional don't-care subspace.
Invalid codewords satisfying the \(D=4\) class condition are allowed to
evolve arbitrarily.


In [15]:

def apply_promise_aware_rotations(qc, regs, alpha):
    d = regs["d"]
    w = regs["w"]
    g = regs["g"]

    side = d[JIDX[3]]
    active = w[6]

    g1, gquarter, ghalf = g

    # D = 0, g = 1: physical dark states exist.
    qc.mcrx(-2.0 * alpha, [active, g1], side)

    # D = 2, g = 1/2: physical dark states exist.
    qc.mcrx(-1.0 * alpha, [active, ghalf], side)

    # D = 4, g = 1/4:
    # ALL PHYSICAL STATES IN THIS CLASS ARE BRIGHT.
    # The active control is deliberately omitted.
    qc.crx(-0.5 * alpha, gquarter, side)



## 11. Assemble one plaquette step


In [16]:

def optimized_plaquette_dontcare_low_t(dt, gs, gt, a):
    alpha = 2.0 * dt / (gs * gt * a)

    qc, regs = make_registers()

    # 1. Bright/color basis.
    forward_color_transform(qc, regs)

    # 2. Physical bright selector.
    compute_domain_walls(qc, regs)
    compute_errors(qc, regs)
    compute_active_flag(qc, regs)

    # 3. Complementary-j pair basis.
    pair_basis_transform(qc, regs)

    # 4. Coupling class from the three relative j bits.
    compute_g_class_relative(qc, regs)

    # 5. Make domain-wall colors invariant across a pair.
    make_domain_colors_invariant_rccx(qc, regs)

    # 6. Residual sign.
    compute_sign_tmp(qc, regs)
    apply_sign_conjugation(qc, regs)

    # 7. Three dynamical angles, with the D=4 active control removed.
    apply_promise_aware_rotations(qc, regs, alpha)

    # 8. Undo sign.
    apply_sign_conjugation(qc, regs)
    uncompute_sign_tmp(qc, regs)

    # 9. Undo pair-label processing.
    undo_domain_colors_invariant_rccx(qc, regs)
    uncompute_g_class_relative(qc, regs)
    inverse_pair_basis_transform(qc, regs)

    # 10. Uncompute physical bright selector.
    uncompute_active_flag(qc, regs)
    uncompute_errors(qc, regs)
    uncompute_domain_walls(qc, regs)

    # 11. Return to the original link basis.
    inverse_color_transform(qc, regs)

    return qc, regs



## 12. Construct one plaquette circuit


In [20]:

dt = 1.0
a  = 1.0
gs = 1.0
gt = 1.0

qc_dontcare, regs = optimized_plaquette_dontcare_low_t(
    dt=dt,
    gs=gs,
    gt=gt,
    a=a,
)

print("Number of qubits:", qc_dontcare.num_qubits)
def fixed_structural_t_count(qc):
    c = qc.count_ops()
    return {
        'rccx': c.get('rccx', 0),
        'ccx': c.get('ccx', 0),
        'explicit_t': c.get('t', 0),
        'explicit_tdg': c.get('tdg', 0),
        'fixed_T_estimate': (
            4*c.get('rccx', 0)
            + 7*c.get('ccx', 0)
            + c.get('t', 0)
            + c.get('tdg', 0)
        ),
    }

fixed_structural_t_count(qc_dontcare)

Number of qubits: 34


{'rccx': 56,
 'ccx': 8,
 'explicit_t': 8,
 'explicit_tdg': 8,
 'fixed_T_estimate': 296}

In [21]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.transpiler import generate_preset_clifford_t_pass_manager


def intermediate_exact_decomposition(qc):
    """
    Decompose controlled rotations, CCX, RCCX, etc.,
    but LEAVE arbitrary Rz rotations unsynthesized.

    This lets us distinguish:
      1. already-explicit T/Tdg gates
      2. arbitrary-angle rotations
    """

    return transpile(
        qc,
        basis_gates=[
            "rz",
            "sx",
            "x",
            "h",
            "s",
            "sdg",
            "t",
            "tdg",
            "cx",
        ],
        optimization_level=0
    )

mid = intermediate_exact_decomposition(qc_dontcare)

print(mid.count_ops())

OrderedDict({'cx': 294, 'h': 166, 't': 152, 'tdg': 144, 'rz': 60, 'x': 50, 'sx': 24, 'sdg': 8, 's': 8})


In [ ]:
mid.print("mid")


## 13. Fixed structural \(T\)-count

Before approximating the continuous-angle `crx`/`mcrx` gates, count

- `RCCX` as 4 \(T/T^\dagger\);
- exact `CCX` as 7 \(T/T^\dagger\);
- explicit `t` and `tdg` gates directly.

For this circuit the expected fixed result is

\[
\boxed{296}.
\]

The three dynamical rotations are deliberately excluded here because their
\(T\)-count depends on the requested synthesis error.


In [22]:

def fixed_structural_t_count(qc):
    c = qc.count_ops()

    n_rccx = c.get("rccx", 0)
    n_ccx  = c.get("ccx", 0)
    n_t    = c.get("t", 0)
    n_tdg  = c.get("tdg", 0)

    fixed = 4*n_rccx + 7*n_ccx + n_t + n_tdg

    return {
        "rccx": n_rccx,
        "ccx": n_ccx,
        "explicit_t": n_t,
        "explicit_tdg": n_tdg,
        "fixed_T_estimate": fixed,
    }


fixed_stats = fixed_structural_t_count(qc_dontcare)
print(fixed_stats)

assert fixed_stats["fixed_T_estimate"] == 296


{'rccx': 56, 'ccx': 8, 'explicit_t': 8, 'explicit_tdg': 8, 'fixed_T_estimate': 296}


In [23]:
print('Number of qubits:', qc_dontcare.num_qubits)
print('Circuit size:', qc_dontcare.size())
print('Circuit depth:', qc_dontcare.depth())
print('High-level counts:')
print(qc_dontcare.count_ops())

Number of qubits: 34
Circuit size: 237
Circuit depth: 89
High-level counts:
OrderedDict({'rccx': 56, 'cx': 54, 'x': 50, 'h': 16, 'cz': 10, 'ccx': 8, 'sdg': 8, 'tdg': 8, 't': 8, 's': 8, 'cu': 6, 'z': 4, 'crx': 1})



## 14. Comparison with the preceding optimization level

| resource | previous low-\(T\) circuit | promise-aware circuit |
|---|---:|---:|
| data qubits | 12 | 12 |
| total qubits | 35 | 34 |
| fixed structural \(T\) | 328 | **296** |
| \(g=1\) rotation | \(C^2R_x\) | \(C^2R_x\) |
| \(g=\tfrac12\) rotation | \(C^2R_x\) | \(C^2R_x\) |
| \(g=\tfrac14\) rotation | \(C^2R_x\) | **\(CR_x\)** |
| identity required on invalid codewords | yes | **no** |

The final Clifford+\(T\) advantage is larger than the fixed 32-\(T\)
difference because one control is also removed from an arbitrary-angle
rotation.



## 15. Optional compilation to Clifford+\(T\)

The result depends on Qiskit version and on the rotation-synthesis error.
Use the same error when comparing optimization levels.


In [24]:

def compile_to_clifford_t(qc, rz_error=1e-6, optimization_level=3):
    from qiskit.transpiler import generate_preset_clifford_t_pass_manager

    pm = generate_preset_clifford_t_pass_manager(
        basis_gates=["cx", "h", "s", "sdg", "t", "tdg"],
        optimization_level=optimization_level,
        rz_synthesis_config={
            "rz_synthesis_error": rz_error,
            "rz_cache_error": rz_error / 10.0,
        },
        seed_transpiler=12345,
    )

    return pm.run(qc)


# Example:

qc_CT = compile_to_clifford_t(
     qc_dontcare,
     rz_error=1e-6,
     optimization_level=3,
 )
#
counts_CT = qc_CT.count_ops()
#
print(counts_CT)
print("T count:",
       counts_CT.get("t", 0) + counts_CT.get("tdg", 0))
print("CX count:", counts_CT.get("cx", 0))
print("Total gates:", qc_CT.size())
print("Depth:", qc_CT.depth())


OrderedDict({'h': 1155, 't': 1027, 's': 504, 'cx': 294, 'tdg': 135, 'sdg': 92})
T count: 1162
CX count: 294
Total gates: 3207
Depth: 2549


In [27]:
# ============================================================
# Export the final compiled Clifford+T plaquette circuit
# ============================================================

from pathlib import Path
import json
import hashlib
import qiskit

from qiskit import qpy, qasm3


# ------------------------------------------------------------
# Choose the circuit to distribute.
#
# qc_CT is the fully synthesized Clifford+T circuit.
# ------------------------------------------------------------

qc_export = qc_CT

outdir = Path("plaquette_circuit_export")
outdir.mkdir(exist_ok=True)


# ------------------------------------------------------------
# 1. QPY: preferred lossless Qiskit representation
# ------------------------------------------------------------

qpy_file = outdir / "SU2_plaquette_jmax_half_CT.qpy"

with open(qpy_file, "wb") as f:
    qpy.dump(qc_export, f)


# ------------------------------------------------------------
# 2. OpenQASM 3: portable textual representation
# ------------------------------------------------------------

qasm_file = outdir / "SU2_plaquette_jmax_half_CT.qasm"

with open(qasm_file, "w") as f:
    qasm3.dump(qc_export, f)


# ------------------------------------------------------------
# 3. Metadata needed to reproduce the compilation
# ------------------------------------------------------------

counts = qc_export.count_ops()

metadata = {
    "description":
        "SU(2) jmax=1/2 promise-aware plaquette evolution, "
        "compiled to Clifford+T",

    "qiskit_version": qiskit.__version__,

    "physics_parameters": {
        "dt": float(dt),
        "a": float(a),
        "gs": float(gs),
        "gt": float(gt),
    },

    "compilation": {
        "basis_gates": ["cx", "h", "s", "sdg", "t", "tdg"],
        "rz_synthesis_error": 1e-6,
        "optimization_level": 3,
        "seed_transpiler": 12345,
    },

    "resources": {
        "num_qubits": qc_export.num_qubits,
        "num_gates": qc_export.size(),
        "depth": qc_export.depth(),
        "T_count":
            int(counts.get("t", 0) + counts.get("tdg", 0)),
        "CX_count":
            int(counts.get("cx", 0)),
        "gate_counts":
            {str(k): int(v) for k, v in counts.items()},
    },

    "qubit_registers": [
        {
            "name": reg.name,
            "size": reg.size,
        }
        for reg in qc_export.qregs
    ],
}

metadata_file = outdir / "SU2_plaquette_jmax_half_CT_metadata.json"

with open(metadata_file, "w") as f:
    json.dump(metadata, f, indent=2)


# ------------------------------------------------------------
# 4. Checksums: useful when distributing the files
# ------------------------------------------------------------

def sha256(filename):
    h = hashlib.sha256()
    with open(filename, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


print("Exported files:")
print("  ", qpy_file)
print("  ", qasm_file)
print("  ", metadata_file)

print("\nSHA256:")
print("QPY  :", sha256(qpy_file))
print("QASM :", sha256(qasm_file))

print("\nCircuit resources:")
print("qubits =", qc_export.num_qubits)
print("gates  =", qc_export.size())
print("depth  =", qc_export.depth())
print("T/Tdg  =", counts.get("t", 0) + counts.get("tdg", 0))
print("CX     =", counts.get("cx", 0))
print("\nGate counts:")
print(counts)

PermissionError: [Errno 1] Operation not permitted: 'plaquette_circuit_export/SU2_plaquette_jmax_half_CT.qpy'


# 16. Classical exhaustive verification of the 625-state promise

The remaining cells use NumPy only.  They reconstruct the exact
\(625\times625\) plaquette operator and verify:

1. there are 41 rank-two bright blocks;
2. the transformed physical space contains 584 labels:
   543 singleton dark labels and 41 two-state bright labels;
3. **all \(D=4\) physical states belong to bright pairs**;
4. the swapped-control `RCCX` domain transform preserves every bright-pair
   permutation and changes the phase only by \(-1\) in the \(D=2\) class;
5. an invalid codeword exists for which the new \(D=4\) rotation is enabled
   even though `active=0`, demonstrating that identity on the unused
   subspace is no longer enforced.


In [15]:

TOL = 1e-12
LINK_CODE = np.array([0, 4, 5, 6, 7], dtype=int)


def truncated_link_U():
    U = np.zeros((2, 2, 5, 5), dtype=complex)
    r = 1.0 / np.sqrt(2.0)

    U[0, 0][0, 1] = r
    U[0, 0][4, 0] = r

    U[0, 1][0, 2] = r
    U[0, 1][3, 0] = -r

    U[1, 0][0, 3] = r
    U[1, 0][2, 0] = -r

    U[1, 1][0, 4] = r
    U[1, 1][1, 0] = r

    return U


def kron4(A1, A2, A3, A4):
    return np.kron(A4, np.kron(A3, np.kron(A2, A1)))


def plaquette_B_physical():
    U = truncated_link_U()
    W = np.zeros((5**4, 5**4), dtype=complex)

    for aa in range(2):
        for bb in range(2):
            for cc in range(2):
                for dd in range(2):
                    W += kron4(
                        U[aa, bb],
                        U[bb, cc],
                        U[dd, cc].conj().T,
                        U[aa, dd].conj().T,
                    )

    return 0.5 * (W + W.conj().T)


def physical_to_qubit_index(p):
    s0 = p % 5
    p //= 5

    s1 = p % 5
    p //= 5

    s2 = p % 5
    p //= 5

    s3 = p % 5

    return int(
        LINK_CODE[s0]
        | (LINK_CODE[s1] << 3)
        | (LINK_CODE[s2] << 6)
        | (LINK_CODE[s3] << 9)
    )


def find_bright_blocks(B=None, tol=TOL):
    if B is None:
        B = plaquette_B_physical()

    N = B.shape[0]
    adj = [set() for _ in range(N)]

    rows, cols = np.nonzero(np.abs(B) > tol)

    for i, j in zip(rows, cols):
        if i != j:
            adj[i].add(j)
            adj[j].add(i)

    seen = set()
    components = []

    for i in range(N):
        if i in seen:
            continue

        stack = [i]
        seen.add(i)
        comp = []

        while stack:
            v = stack.pop()
            comp.append(v)

            for w in adj[v]:
                if w not in seen:
                    seen.add(w)
                    stack.append(w)

        components.append(sorted(comp))

    blocks = []

    for comp in components:
        if len(comp) == 1:
            continue

        color = {comp[0]: 0}
        stack = [comp[0]]

        while stack:
            v = stack.pop()

            for w in adj[v]:
                if w not in color:
                    color[w] = 1 - color[v]
                    stack.append(w)

        side_A = sorted(x for x in comp if color[x] == 0)
        side_B = sorted(x for x in comp if color[x] == 1)

        C = B[np.ix_(side_A, side_B)].real
        u, s, vh = np.linalg.svd(C, full_matrices=False)

        assert np.sum(s > tol) == 1

        blocks.append({
            "dimension": len(comp),
            "A_states": [physical_to_qubit_index(x) for x in side_A],
            "A_amp": u[:, 0].copy(),
            "B_states": [physical_to_qubit_index(x) for x in side_B],
            "B_amp": vh[0, :].copy(),
            "g": float(s[0]),
        })

    assert len(blocks) == 41

    return blocks



## 17. Sparse classical gate simulator

This simulator is used only for verification.  It includes the exact
Qiskit `RCCX` relative phases.


In [ ]:

def sp_cx(state, c, t):
    out = {}
    mc = 1 << c
    mt = 1 << t

    for k, amp in state.items():
        kk = k ^ mt if (k & mc) else k
        out[kk] = out.get(kk, 0.0) + amp

    return out


def sp_ccx(state, c1, c2, t):
    out = {}
    m1 = 1 << c1
    m2 = 1 << c2
    mt = 1 << t

    for k, amp in state.items():
        kk = k ^ mt if ((k & m1) and (k & m2)) else k
        out[kk] = out.get(kk, 0.0) + amp

    return out


def sp_rccx(state, c1, c2, t):
    # Qiskit RCCX convention:
    # controls c1,c2 and target t.
    out = {}

    m1 = 1 << c1
    m2 = 1 << c2
    mt = 1 << t

    for k, amp in state.items():
        b1 = bool(k & m1)
        b2 = bool(k & m2)
        bt = bool(k & mt)

        kk = k
        phase = 1.0 + 0.0j

        if b1 and (not b2) and bt:
            phase = -1.0

        elif b1 and b2:
            kk = k ^ mt
            phase = -1.0j if bt else 1.0j

        out[kk] = out.get(kk, 0.0) + phase * amp

    return out


def sp_z(state, q):
    mq = 1 << q

    return {
        k: (-amp if (k & mq) else amp)
        for k, amp in state.items()
    }


def sp_cc_h(state, c1, c2, target):
    out = {}

    m1 = 1 << c1
    m2 = 1 << c2
    mt = 1 << target
    r = 1.0 / np.sqrt(2.0)

    for k, amp in state.items():
        if (k & m1) and (k & m2):
            bit = 1 if (k & mt) else 0

            k0 = k & ~mt
            k1 = k0 | mt

            out[k0] = out.get(k0, 0.0) + r * amp
            out[k1] = out.get(k1, 0.0) + (-r if bit else r) * amp

        else:
            out[k] = out.get(k, 0.0) + amp

    return {
        k: v
        for k, v in out.items()
        if abs(v) > 1e-12
    }


def classical_forward_color(state):
    for link in (2, 3):
        state = sp_cx(state, qj(link), qm(link))
        state = sp_cx(state, qj(link), qmp(link))

    for x, y, jx, jy in VERTEX_SPEC:
        state = sp_cx(state, x, y)
        state = sp_ccx(state, jx, y, x)

    state = sp_z(state, VERTEX_SPEC[0][0])
    state = sp_z(state, VERTEX_SPEC[2][0])

    for x, y, jx, jy in VERTEX_SPEC:
        state = sp_cc_h(state, jx, jy, x)

    return state


def domain_walls_from_root(root):
    j = [(root >> q) & 1 for q in JIDX]

    return (
        j[3] ^ j[0],
        j[0] ^ j[1],
        j[1] ^ j[2],
        j[2] ^ j[3],
    )


def active_from_root(root):
    p = domain_walls_from_root(root)

    xs = [
        (root >> spec[0]) & 1
        for spec in VERTEX_SPEC
    ]

    ys = [
        (root >> spec[1]) & 1
        for spec in VERTEX_SPEC
    ]

    errors = [
        (1 - p[i]) & ys[i]
        for i in range(4)
    ]

    return (sum(xs) == 0) and (sum(errors) == 0)



## 18. Physical transformed roots and the \(D=4\) promise


In [ ]:

B = plaquette_B_physical()
blocks = find_bright_blocks(B)

# All 625 valid encoded computational states.
valid_encoded = []

for s0 in range(5):
    for s1 in range(5):
        for s2 in range(5):
            for s3 in range(5):
                q = (
                    int(LINK_CODE[s0])
                    | (int(LINK_CODE[s1]) << 3)
                    | (int(LINK_CODE[s2]) << 6)
                    | (int(LINK_CODE[s3]) << 9)
                )
                valid_encoded.append(q)

valid_encoded = set(valid_encoded)
assert len(valid_encoded) == 625

# Union of computational roots after the exact bright/color transform.
physical_roots = set()

for q in valid_encoded:
    out = classical_forward_color({q: 1.0})
    physical_roots.update(out.keys())

assert len(physical_roots) == 625

# The 82 bright roots.
bright_roots = set()

for block in blocks:
    A = {
        int(q): complex(a)
        for q, a in zip(block["A_states"], block["A_amp"])
    }

    BB = {
        int(q): complex(a)
        for q, a in zip(block["B_states"], block["B_amp"])
    }

    Ao = classical_forward_color(A)
    Bo = classical_forward_color(BB)

    assert len(Ao) == 1
    assert len(Bo) == 1

    bright_roots.add(next(iter(Ao)))
    bright_roots.add(next(iter(Bo)))

assert len(bright_roots) == 82

D4_physical = {
    r for r in physical_roots
    if sum(domain_walls_from_root(r)) == 4
}

D4_bright = D4_physical & bright_roots
D4_dark = D4_physical - bright_roots

print("physical roots:", len(physical_roots))
print("bright roots:", len(bright_roots))
print("D=4 physical roots:", len(D4_physical))
print("D=4 bright roots:", len(D4_bright))
print("D=4 physical dark roots:", len(D4_dark))

assert len(D4_physical) == 32
assert len(D4_bright) == 32
assert len(D4_dark) == 0



The last assertion is the mathematical reason the \(g=\tfrac14\) rotation
does not need `active` on the physical code space.



## 19. Verify the swapped-control RCCX domain transform

The exact and RCCX versions must:

- map every physical bright pair to the same two computational roots;
- differ in relative phase by \(-1\) only for \(g=\tfrac12\).


In [ ]:

def pair_domain_transform(root, amp, p, use_rccx=False):
    state = {root: amp}
    side = JIDX[3]

    # Pair basis.
    for k in (0, 1, 2):
        state = sp_cx(state, side, JIDX[k])

    # Append four classical p bits virtually at positions 12..15.
    extended = {}

    for q, a in state.items():
        qq = q

        for i, pi in enumerate(p):
            if pi:
                qq |= 1 << (12 + i)

        extended[qq] = a

    state = extended

    for i in range(4):
        yi = VERTEX_SPEC[i][1]

        if use_rccx:
            # Swapped control ordering: (p_i, side, y_i)
            state = sp_rccx(state, 12 + i, side, yi)
        else:
            state = sp_ccx(state, side, 12 + i, yi)

    assert len(state) == 1

    q, a = next(iter(state.items()))

    return q & ((1 << 12) - 1), a


phase_ratio_counts = {}
max_root_mismatch = 0

for block in blocks:
    A = {
        int(q): complex(a)
        for q, a in zip(block["A_states"], block["A_amp"])
    }

    BB = {
        int(q): complex(a)
        for q, a in zip(block["B_states"], block["B_amp"])
    }

    Ao = classical_forward_color(A)
    Bo = classical_forward_color(BB)

    ra, pa = next(iter(Ao.items()))
    rb, pb = next(iter(Bo.items()))

    p = domain_walls_from_root(ra)
    D = sum(p)

    ea, epa = pair_domain_transform(ra, pa, p, use_rccx=False)
    eb, epb = pair_domain_transform(rb, pb, p, use_rccx=False)

    ra2, rpa = pair_domain_transform(ra, pa, p, use_rccx=True)
    rb2, rpb = pair_domain_transform(rb, pb, p, use_rccx=True)

    assert ea == ra2
    assert eb == rb2

    c_exact = epa * np.conj(epb)
    c_rccx  = rpa * np.conj(rpb)

    ratio = c_rccx / c_exact

    expected = -1.0 if D == 2 else 1.0

    assert abs(ratio - expected) < 1e-12

    phase_ratio_counts[(D, expected)] = (
        phase_ratio_counts.get((D, expected), 0) + 1
    )

print(phase_ratio_counts)

assert phase_ratio_counts[(0, 1.0)] == 1
assert phase_ratio_counts[(2, -1.0)] == 24
assert phase_ratio_counts[(4, 1.0)] == 16



## 20. Show explicitly that the unused subspace is no longer protected

We search the 3471 invalid codewords for one that reaches the \(D=4\)
selector with `active=0`.

The previous circuit suppressed the dynamical rotation on such a component.
The new circuit intentionally does not.


In [ ]:

invalid_example = None

for q in range(2**12):
    if q in valid_encoded:
        continue

    out = classical_forward_color({q: 1.0})

    for root, amp in out.items():
        D = sum(domain_walls_from_root(root))

        if D == 4 and (not active_from_root(root)):
            invalid_example = (q, root, amp)
            break

    if invalid_example is not None:
        break


assert invalid_example is not None

q_invalid, transformed_root, amp = invalid_example

print("Example invalid input integer:", q_invalid)
print("Invalid input bitstring:", format(q_invalid, "012b"))
print("Transformed root:", format(transformed_root, "012b"))
print("D =", sum(domain_walls_from_root(transformed_root)))
print("active =", active_from_root(transformed_root))
print("amplitude =", amp)

print(
    "\nIn the promise-aware circuit the g=1/4 CRx is enabled by the D=4 "
    "class flag even though active=0.  This behavior is allowed because "
    "the input is outside the 625-state encoded subspace."
)



# 21. Summary

The optimization implemented here is deliberately conservative about
physical dark states:

- no physical dark-state protection is removed;
- the only selector control discarded is one proven redundant on the encoded
  physical space;
- the domain-color relative-phase optimization is verified block by block.

The next \(T\)-count target would be the remaining `active` computation and
the two doubly-controlled arbitrary rotations.  Reducing those further
requires either a more aggressive incomplete Boolean synthesis or
measurement-assisted temporary logical-AND uncomputation.
